In [1]:
import pandas as pd
import numpy as np
import os

# Création des dossiers

os.makedirs(
    "../data/processed",
    exist_ok=True
)

os.makedirs(
    "../data/rejects",
    exist_ok=True
)

# Chargement des datasets

sales = pd.read_csv(
    "../data/raw/internal/ventes_brutes.csv"
)

weather = pd.read_csv(
    "../data/raw/open_data/weather.csv"
)

inflation = pd.read_csv(
    "../data/raw/open_data/inflation.csv"
)

gdp = pd.read_csv(
    "../data/raw/open_data/gdp.csv"
)

interest = pd.read_csv(
    "../data/raw/open_data/interest_rate.csv"
)

wheat = pd.read_csv(
    "../data/raw/open_data/wheat_price.csv"
)

corn = pd.read_csv(
    "../data/raw/open_data/corn_price.csv"
)

soybean = pd.read_csv(
    "../data/raw/open_data/soybean_price.csv"
)

datasets = {
    "sales": sales,
    "weather": weather,
    "inflation": inflation,
    "gdp": gdp,
    "interest": interest,
    "wheat": wheat,
    "corn": corn,
    "soybean": soybean
}

for name, df in datasets.items():
    print(name, df.shape)

sales (28451, 11)
weather (7306, 4)
inflation (20, 3)
gdp (20, 3)
interest (240, 3)
wheat (120, 2)
corn (120, 2)
soybean (120, 2)


In [2]:
# Vérification des datasets
for name, df in datasets.items():

    print(name.upper())

    print("Colonnes :", df.columns.tolist())

    print("\nValeurs manquantes :")
    print(df.isnull().sum())

SALES
Colonnes : ['transaction_id', 'date_vente', 'categorie_produit_1', 'categorie_produit_2', 'produit', 'quantite_vendue', 'trans_amount_weight', 'region', 'pays', 'code_pays', 'devise']

Valeurs manquantes :
transaction_id         0
date_vente             0
categorie_produit_1    0
categorie_produit_2    0
produit                0
quantite_vendue        0
trans_amount_weight    0
region                 0
pays                   0
code_pays              0
devise                 0
dtype: int64
WEATHER
Colonnes : ['date', 'temperature', 'precipitation', 'code_pays']

Valeurs manquantes :
date             0
temperature      0
precipitation    0
code_pays        0
dtype: int64
INFLATION
Colonnes : ['code_pays', 'date', 'inflation']

Valeurs manquantes :
code_pays    0
date         0
inflation    0
dtype: int64
GDP
Colonnes : ['code_pays', 'date', 'gdp_growth']

Valeurs manquantes :
code_pays     0
date          0
gdp_growth    0
dtype: int64
INTEREST
Colonnes : ['date', 'code_pays', 'int

In [3]:
# Préparation des dates

sales["date_vente"] = pd.to_datetime(
    sales["date_vente"],
    errors="coerce"
)

weather["date"] = pd.to_datetime(
    weather["date"],
    errors="coerce"
)

interest["date"] = pd.to_datetime(
    interest["date"],
    errors="coerce"
)

wheat["date"] = pd.to_datetime(
    wheat["date"],
    errors="coerce"
)

corn["date"] = pd.to_datetime(
    corn["date"],
    errors="coerce"
)

soybean["date"] = pd.to_datetime(
    soybean["date"],
    errors="coerce"
)

# Année pour l'inflation et le PIB

inflation["annee"] = pd.to_numeric(
    inflation["date"],
    errors="coerce"
)

gdp["annee"] = pd.to_numeric(
    gdp["date"],
    errors="coerce"
)

# Année et mois des ventes

sales["annee"] = sales["date_vente"].dt.year
sales["mois"] = sales["date_vente"].dt.month

sales["date_mois"] = (
    sales["date_vente"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Année et mois des données mensuelles

for df in [weather, interest, wheat, corn, soybean]:

    df["annee"] = df["date"].dt.year
    df["mois"] = df["date"].dt.month

# Suppression des doublons uniquement dans les Open Data

weather = weather.drop_duplicates()
inflation = inflation.drop_duplicates()
gdp = gdp.drop_duplicates()
interest = interest.drop_duplicates()
wheat = wheat.drop_duplicates()
corn = corn.drop_duplicates()
soybean = soybean.drop_duplicates()

print("Préparation des dates terminée")
print("Période :", sales["date_vente"].min(), "au", sales["date_vente"].max())
print("Dimensions des ventes :", sales.shape)

Préparation des dates terminée
Période : 2007-01-03 00:00:00 au 2016-12-22 00:00:00
Dimensions des ventes : (28451, 14)


In [4]:
# Agrégation et fusion des données

# Agrégation mensuelle de la météo par pays

weather_monthly = (
    weather
    .groupby(
        ["code_pays", "annee", "mois"],
        as_index=False
    )
    [
        ["temperature", "precipitation"]
    ]
    .mean()
)

# Agrégation mensuelle des taux d'intérêt par pays

interest_monthly = (
    interest
    .groupby(
        ["code_pays", "annee", "mois"],
        as_index=False
    )
    [
        ["interest_rate"]
    ]
    .mean()
)

# Agrégation mensuelle des prix agricoles

wheat_monthly = (
    wheat
    .groupby(
        ["annee", "mois"],
        as_index=False
    )
    [
        ["wheat_price"]
    ]
    .mean()
)

corn_monthly = (
    corn
    .groupby(
        ["annee", "mois"],
        as_index=False
    )
    [
        ["corn_price"]
    ]
    .mean()
)

soybean_monthly = (
    soybean
    .groupby(
        ["annee", "mois"],
        as_index=False
    )
    [
        ["soybean_price"]
    ]
    .mean()
)

# Fusion principale

dataset = sales.copy()

dataset = dataset.merge(
    inflation[
        ["code_pays", "annee", "inflation"]
    ],
    on=["code_pays", "annee"],
    how="left"
)

dataset = dataset.merge(
    gdp[
        ["code_pays", "annee", "gdp_growth"]
    ],
    on=["code_pays", "annee"],
    how="left"
)

dataset = dataset.merge(
    weather_monthly,
    on=["code_pays", "annee", "mois"],
    how="left"
)

dataset = dataset.merge(
    interest_monthly,
    on=["code_pays", "annee", "mois"],
    how="left"
)

dataset = dataset.merge(
    wheat_monthly,
    on=["annee", "mois"],
    how="left"
)

dataset = dataset.merge(
    corn_monthly,
    on=["annee", "mois"],
    how="left"
)

dataset = dataset.merge(
    soybean_monthly,
    on=["annee", "mois"],
    how="left"
)

open_data_cols = [
    "inflation",
    "gdp_growth",
    "temperature",
    "precipitation",
    "interest_rate",
    "wheat_price",
    "corn_price",
    "soybean_price"
]

print("Fusion terminée")
print("Dimensions :", dataset.shape)

print("\nValeurs manquantes après fusion :")
print(dataset[open_data_cols].isnull().sum())

Fusion terminée
Dimensions : (28451, 22)

Valeurs manquantes après fusion :
inflation        0
gdp_growth       0
temperature      0
precipitation    0
interest_rate    0
wheat_price      0
corn_price       0
soybean_price    0
dtype: int64


In [5]:
# Nettoyage final

# Identification des lignes invalides

condition_rejet = (
    dataset["date_vente"].isna()
    |
    dataset["quantite_vendue"].isna()
    |
    (dataset["quantite_vendue"] <= 0)
    |
    dataset[open_data_cols].isna().any(axis=1)
)

# Création du fichier des rejets

rejets = dataset[
    condition_rejet
].copy()

rejets["motif_rejet"] = np.select(
    [
        rejets["date_vente"].isna(),
        rejets["quantite_vendue"].isna(),
        rejets["quantite_vendue"] <= 0,
        rejets[open_data_cols].isna().any(axis=1)
    ],
    [
        "Date de vente manquante",
        "Quantité manquante",
        "Quantité inférieure ou égale à zéro",
        "Données Open Data manquantes"
    ],
    default="Autre erreur"
)

# Conservation des lignes valides

dataset = dataset[
    ~condition_rejet
].copy()

# Variables temporelles

dataset["trimestre"] = (
    dataset["date_vente"].dt.quarter
)

dataset["jour_semaine"] = (
    dataset["date_vente"].dt.dayofweek
)

# Tri du dataset

dataset = dataset.sort_values(
    [
        "date_vente",
        "code_pays",
        "produit"
    ]
)

dataset = dataset.reset_index(drop=True)

# Enregistrement des rejets

rejets.to_csv(
    "../data/rejects/rejets_preparation.csv",
    index=False
)

print("Nettoyage terminé")
print("Lignes valides :", len(dataset))
print("Lignes rejetées :", len(rejets))

print("\nValeurs manquantes :")
print(dataset.isnull().sum())

Nettoyage terminé
Lignes valides : 28451
Lignes rejetées : 0

Valeurs manquantes :
transaction_id         0
date_vente             0
categorie_produit_1    0
categorie_produit_2    0
produit                0
quantite_vendue        0
trans_amount_weight    0
region                 0
pays                   0
code_pays              0
devise                 0
annee                  0
mois                   0
date_mois              0
inflation              0
gdp_growth             0
temperature            0
precipitation          0
interest_rate          0
wheat_price            0
corn_price             0
soybean_price          0
trimestre              0
jour_semaine           0
dtype: int64


In [6]:
# Vérification du périmètre

print(
    "Secteur :",
    dataset["categorie_produit_1"].unique()
)

print(
    "Région :",
    dataset["region"].unique()
)

print(
    "Pays :",
    dataset["code_pays"].unique()
)

print(
    "Période :",
    dataset["date_vente"].min(),
    "au",
    dataset["date_vente"].max()
)

print(
    "Transactions uniques :",
    dataset["transaction_id"].nunique()
)

print(
    "Lignes rejetées :",
    len(rejets)
)

# Vérifications

assert (
    dataset["categorie_produit_1"]
    .str.strip()
    .str.lower()
    .eq("agriculture")
    .all()
)

assert (
    dataset["region"]
    .str.strip()
    .str.lower()
    .eq("northern america")
    .all()
)

assert (
    dataset["code_pays"]
    .str.strip()
    .str.upper()
    .isin(["USA", "CAN"])
    .all()
)

assert (
    dataset[open_data_cols]
    .isnull()
    .sum()
    .sum()
    == 0
)

assert (
    dataset["transaction_id"]
    .notna()
    .all()
)

assert (
    dataset["transaction_id"]
    .is_unique
)

assert (
    len(dataset) + len(rejets)
    == len(sales)
)

# Sauvegarde du dataset final

output_path = (
    "../data/processed/dataset_final.csv"
)

dataset.to_csv(
    output_path,
    index=False
)

print("\nDataset final sauvegardé :", output_path)
print("Fichier créé :", os.path.exists(output_path))
print("Dimensions finales :", dataset.shape)

dataset.head()

Secteur : ['agriculture']
Région : ['Northern America']
Pays : ['USA' 'CAN']
Période : 2007-01-03 00:00:00 au 2016-12-22 00:00:00
Transactions uniques : 28451
Lignes rejetées : 0



Dataset final sauvegardé : ../data/processed/dataset_final.csv
Fichier créé : True
Dimensions finales : (28451, 24)


,transaction_id,date_vente,categorie_produit_1,categorie_produit_2,produit,quantite_vendue,trans_amount_weight,region,pays,code_pays,...,inflation,gdp_growth,temperature,precipitation,interest_rate,wheat_price,corn_price,soybean_price,trimestre,jour_semaine
0,319041,2007-01-03,agriculture,soil & crop treatment,sprayer (self propelled),8.0,24.0,Northern America,united states,USA,...,2.852672,2.003858,-2.735484,0.632258,5.32,172.566631,165.160019,255.867379,1,2
1,319042,2007-01-03,agriculture,soil & crop treatment,sprayer (self propelled),4.0,12.0,Northern America,united states,USA,...,2.852672,2.003858,-2.735484,0.632258,5.32,172.566631,165.160019,255.867379,1,2
2,319043,2007-01-03,agriculture,soil & crop treatment,sprayer (self propelled),10.0,30.0,Northern America,united states,USA,...,2.852672,2.003858,-2.735484,0.632258,5.32,172.566631,165.160019,255.867379,1,2
3,319103,2007-01-03,agriculture,soil & crop treatment,sprayer (self propelled),1.0,3.0,Northern America,united states,USA,...,2.852672,2.003858,-2.735484,0.632258,5.32,172.566631,165.160019,255.867379,1,2
4,319128,2007-01-03,agriculture,soil & crop treatment,sprayer (self propelled),1.0,4.0,Northern America,united states,USA,...,2.852672,2.003858,-2.735484,0.632258,5.32,172.566631,165.160019,255.867379,1,2
